In [0]:
%pip install torch

In [0]:
dbutils.library.restartPython()

In [0]:
import torch  
import torch.nn as nn  
import torch.optim as optim  
from torch.utils.data import Dataset, DataLoader  
import random  
import numpy as np  
import matplotlib.pyplot as plt  
import pandas as pd
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
from mlflow.exceptions import RestException
from mlflow.models import infer_signature

# ---- Parameters ----  
TCN_CHANNELS = 64  
TCN_LAYERS = 3  
HIDDEN_DIM = 64  
BATCH_SIZE = 32  
EPOCHS = 20  
POOL_FACTOR = 4  
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  

# ---- Model Registry Configuration ----
MODEL_NAME = "acubed.ffr.ffr_difficulty_model"  # Use acubed catalog

# ---- Load Data Directly from Table ----
df = spark.table('acubed.ffr.gold__features').toPandas()

# Filter to difficulty > 0
df = df[df['difficulty'] > 0].copy()

# Define ALL 16 feature columns (updated to match gold layer schema)
FEATURE_COLS = [
    # Density Features (3)
    'vertical_density', 'horizontal_density', 'position_in_song',
    # Directional Complexity Features (4)
    'orientation_changed', 'is_single_arrow', 'double_jump', 'triple_quad_jump',
    # Temporal Pattern Features (3)
    'note_spacing', 'is_burst_note', 'is_isolated_note',
    # Song-Level Features (4)
    'song_length_log_normalized', 'vertical_density_cv', 'horizontal_density_cv', 'unique_orientations',
    # Game Mechanics (2)
    'hold', 'mine'
]
NUM_FEATURES = len(FEATURE_COLS)

print(f"Loaded {len(df)} notes from {df['song_id'].nunique()} songs")
print(f"Using {NUM_FEATURES} features: {FEATURE_COLS}")

# Group by song_id and create sequences
song_groups = df.groupby('song_id')

# ---- Real Data Variable-Length Dataset ----  
class RealDataSequenceDataset(Dataset):  
    def __init__(self, df, feature_cols, song_groups):
        self.data = []
        self.labels = []
        self.lengths = []
        self.song_ids = []
        
        for song_id, group in song_groups:
            # Sort by note_id in increasing order
            group_sorted = group.sort_values('note_id')
            
            # Extract feature sequence
            seq = group_sorted[feature_cols].values.astype(np.float32)
            
            # Get difficulty (same for all notes in a song)
            difficulty = group_sorted['difficulty'].iloc[0]
            
            self.data.append(seq)
            self.labels.append(difficulty)
            self.lengths.append(len(seq))
            self.song_ids.append(song_id)
        
        # Get max difficulty for normalization
        self.max_difficulty = max(self.labels)
        print(f"Max difficulty in dataset: {self.max_difficulty}")
        
    def __len__(self):  
        return len(self.data)  
    
    def __getitem__(self, idx):  
        return (torch.from_numpy(self.data[idx]), 
                torch.tensor(self.labels[idx], dtype=torch.float32), 
                self.lengths[idx])

def collate_fn(batch):  
    seqs, ys, lengths = zip(*batch)  
    max_len = max(lengths)  
    # Pad sequences  
    padded_seqs = []  
    mask = []  
    for seq, l in zip(seqs, lengths):  
        pad = torch.zeros(max_len - l, seq.shape[1])  
        padded_seqs.append(torch.cat([seq, pad], dim=0))  
        mask.append(torch.cat([torch.ones(l), torch.zeros(max_len - l)]))  
    padded_seqs = torch.stack(padded_seqs)      # [batch, max_len, num_features]  
    mask = torch.stack(mask)                    # [batch, max_len]  
    ys = torch.stack(ys)                        # [batch]  
    return padded_seqs, ys, mask  

# ---- TCN Block ----  
class TCNBlock(nn.Module):  
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):  
        super().__init__()  
        self.conv = nn.Conv1d(  
            in_channels, out_channels,  
            kernel_size,  
            padding=(kernel_size-1)*dilation,  
            dilation=dilation  
        )  
        self.relu = nn.ReLU()  
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()  
    def forward(self, x):  
        out = self.conv(x)  
        out = self.relu(out)  
        out = out[..., :x.shape[-1]]  # Trim padding for causal conv  
        return out + self.downsample(x)  

# ---- Vectorized Monotonic Attention TCN Model with Downsampling ----  
class MonotonicAttentionTCNModel(nn.Module):  
    def __init__(self, num_features, tcn_channels, tcn_layers, hidden_dim, max_difficulty, pool_factor=4):  
        super().__init__()  
        self.max_difficulty = max_difficulty  
        self.pool_factor = pool_factor  
        # TCN encoder  
        layers = []  
        in_channels = num_features  
        for i in range(tcn_layers):  
            dilation = 2 ** i  
            layers.append(TCNBlock(in_channels, tcn_channels, kernel_size=3, dilation=dilation))  
            in_channels = tcn_channels  
        self.tcn = nn.Sequential(*layers)  
        # Downsampling layer  
        self.pool = nn.AvgPool1d(kernel_size=pool_factor, stride=pool_factor, ceil_mode=True)  
        # Attention  
        self.attention_weights = nn.Parameter(torch.rand(tcn_channels))  
        # Output projection - REMOVED Softplus to allow full range [0, 120]
        self.output_proj = nn.Sequential(  
            nn.Linear(tcn_channels, hidden_dim),  
            nn.ReLU(),  
            nn.Linear(hidden_dim, 1)  # No Softplus here!
        )  

    def forward(self, batch_seq, mask):  
        # batch_seq: [batch, max_len, num_features]  
        # mask: [batch, max_len]  
        batch_seq = batch_seq.to(next(self.parameters()).device)  
        mask = mask.to(batch_seq.device)  
        x = batch_seq.transpose(1,2)  # [batch, num_features, max_len]  
        tcn_out = self.tcn(x)         # [batch, tcn_channels, max_len]  
        # Downsample sequence and mask  
        tcn_out = self.pool(tcn_out)  # [batch, tcn_channels, pooled_len]  
        pooled_len = tcn_out.shape[-1]  
        mask = self.pool(mask.unsqueeze(1)).squeeze(1)   # [batch, pooled_len]  
        mask = (mask > 0).float()  # Convert to binary mask  
        tcn_out = tcn_out.transpose(1,2)  # [batch, pooled_len, tcn_channels]  
        # Attention (vectorized, with mask)  
        attn_scores = torch.matmul(tcn_out, torch.nn.functional.softplus(self.attention_weights))  # [batch, pooled_len]  
        attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))             # Mask out padding  
        attn_weights = torch.softmax(attn_scores, dim=1)                            # [batch, pooled_len]  
        attn_weights = attn_weights * mask                                          # Zero-out padding  
        attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)# Renormalize  
        attended = torch.sum(attn_weights.unsqueeze(-1) * tcn_out, dim=1)           # [batch, tcn_channels]  
        # Output with sigmoid to get [0, 1] range, then scale to [0, max_difficulty]
        out = self.output_proj(attended)  # [batch, 1]  
        y_pred = self.max_difficulty * torch.sigmoid(out).squeeze(-1)  # [batch]  
        return y_pred  

# ---- Training & Evaluation ----  
def train(model, loader, optimizer, criterion):  
    model.train()  
    total_loss = 0  
    total_samples = 0  
    for batch_seqs, batch_ys, batch_mask in loader:  
        optimizer.zero_grad()  
        preds = model(batch_seqs, batch_mask)  
        loss = criterion(preds, batch_ys.to(preds.device))  
        loss.backward()  
        optimizer.step()  
        total_loss += loss.item() * batch_seqs.shape[0]  
        total_samples += batch_seqs.shape[0]  
    return total_loss / total_samples  

def evaluate(model, loader, criterion):  
    model.eval()  
    total_loss = 0  
    total_samples = 0  
    with torch.no_grad():  
        for batch_seqs, batch_ys, batch_mask in loader:  
            preds = model(batch_seqs, batch_mask)  
            loss = criterion(preds, batch_ys.to(preds.device))  
            total_loss += loss.item() * batch_seqs.shape[0]  
            total_samples += batch_seqs.shape[0]  
    return total_loss / total_samples  

# ---- Main ----  
if __name__ == '__main__':  
    # No fixed seed - allows different train/val splits each run for better generalization

    # Prepare dataset with real data
    dataset = RealDataSequenceDataset(df, FEATURE_COLS, song_groups)
    
    train_size = int(0.8 * len(dataset))  
    val_size = len(dataset) - train_size  
    indices = list(range(len(dataset)))  
    random.shuffle(indices)  
    train_indices = indices[:train_size]  
    val_indices = indices[train_size:]  

    train_dataset = torch.utils.data.Subset(dataset, train_indices)  
    val_dataset = torch.utils.data.Subset(dataset, val_indices)  

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)  
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)  

    # ---- Try to Load Existing Model for Fine-Tuning ----
    model_uri = f"models:/{MODEL_NAME}@latest"
    is_finetuning = False
    previous_epochs = 0
    
    try:
        print("\nChecking for existing model with 'latest' alias...")
        client = MlflowClient()
        
        # Get model version and its run info
        model_version_info = client.get_model_version_by_alias(MODEL_NAME, "latest")
        previous_run_id = model_version_info.run_id
        
        # Get previous run's cumulative epochs
        previous_run = client.get_run(previous_run_id)
        previous_epochs = int(previous_run.data.params.get('cumulative_epochs', 0))
        
        # Load model
        model = mlflow.pytorch.load_model(model_uri)
        model = model.to(DEVICE)
        is_finetuning = True
        
        print(f"✓ Found existing model. Will fine-tune from checkpoint.")
        print(f"  Previous cumulative epochs: {previous_epochs}")
        print(f"  Model max_difficulty: {model.max_difficulty}")
    except (RestException, Exception) as e:
        print(f"✗ No existing model found. Training from scratch.")
        print(f"  Reason: {str(e)[:100]}")
        # Create new model from scratch
        model = MonotonicAttentionTCNModel(  
            num_features=NUM_FEATURES,  
            tcn_channels=TCN_CHANNELS,  
            tcn_layers=TCN_LAYERS,  
            hidden_dim=HIDDEN_DIM,  
            max_difficulty=dataset.max_difficulty,
            pool_factor=POOL_FACTOR  
        ).to(DEVICE)  

    optimizer = optim.Adam(model.parameters(), lr=1e-3)  
    criterion = nn.L1Loss()  

    # Calculate cumulative epoch tracking
    epoch_offset = previous_epochs
    new_cumulative_epochs = previous_epochs + EPOCHS

    # ---- Start MLflow Run ----
    mlflow.set_experiment("/Users/wirrywoo.ffr@gmail.com/ffr-difficulty-experiments")
    
    run_name = "tcn-attention-finetuning" if is_finetuning else "tcn-attention-training"
    with mlflow.start_run(run_name=run_name) as run:
        # Log parameters
        mlflow.log_params({
            "tcn_channels": TCN_CHANNELS,
            "tcn_layers": TCN_LAYERS,
            "hidden_dim": HIDDEN_DIM,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "pool_factor": POOL_FACTOR,
            "learning_rate": 1e-3,
            "optimizer": "Adam",
            "loss_function": "L1Loss",
            "num_features": NUM_FEATURES,
            "max_difficulty": dataset.max_difficulty,
            "train_size": train_size,
            "val_size": val_size,
            "is_finetuning": is_finetuning,
            "random_seed": "None (randomized each run)",
            "previous_cumulative_epochs": previous_epochs,
            "cumulative_epochs": new_cumulative_epochs  # Track total epochs across all runs
        })
        
        # ---- Lists for plotting ----  
        train_mae_list = []  
        val_mae_list = []  
        cumulative_epoch_list = []  # Track cumulative epochs for plotting

        # Training loop  
        print(f"\n{'='*60}")
        print(f"{'FINE-TUNING' if is_finetuning else 'TRAINING'} MODEL FOR {EPOCHS} EPOCHS")
        if is_finetuning:
            print(f"Continuing from epoch {epoch_offset} → {new_cumulative_epochs}")
        print(f"{'='*60}\n")
        
        for epoch in range(EPOCHS):  
            train_loss = train(model, train_loader, optimizer, criterion)  
            val_loss = evaluate(model, val_loader, criterion)  
            train_mae_list.append(train_loss)  
            val_mae_list.append(val_loss)  
            
            # Track cumulative epochs for plotting
            cumulative_epoch = epoch_offset + epoch
            cumulative_epoch_list.append(cumulative_epoch)
            
            # Log metrics to MLflow with cumulative epoch as step
            mlflow.log_metrics({
                "train_mae": train_loss,
                "val_mae": val_loss
            }, step=cumulative_epoch)
            print(f"Epoch {cumulative_epoch+1}/{new_cumulative_epochs} - Train MAE: {train_loss:.4f} - Val MAE: {val_loss:.4f}")  

        # Log final metrics
        mlflow.log_metrics({
            "final_train_mae": train_mae_list[-1],
            "final_val_mae": val_mae_list[-1]
        })
        
        # ---- Plot MAE Curve ----  
        plt.figure(figsize=(10, 5))  
        plt.plot(cumulative_epoch_list, train_mae_list, label='Train MAE')  
        plt.plot(cumulative_epoch_list, val_mae_list, label='Validation MAE')  
        plt.xlabel('Cumulative Epoch')  
        plt.ylabel('MAE (Difficulty Units)')  
        plt.title('Training and Validation MAE Over Time')  
        plt.legend()  
        plt.grid()  
        
        # Save plot to MLflow
        mlflow.log_figure(plt.gcf(), "mae_curve.png")
        plt.show()
        
        # ---- Log Model to MLflow Registry ----
        print(f"\nLogging model to MLflow registry: {MODEL_NAME}")
        
        # Create sample input for signature (use first batch from train_loader)
        sample_batch = next(iter(train_loader))
        sample_input = (sample_batch[0], sample_batch[2])  # (sequences, mask)
        
        # Get model prediction for signature
        with torch.no_grad():
            sample_output = model(sample_batch[0], sample_batch[2])
        
        # Infer signature
        signature = infer_signature(
            model_input={"sequences": sample_input[0].numpy(), "mask": sample_input[1].numpy()},
            model_output=sample_output.cpu().numpy()
        )
        
        # Log model with signature
        model_info = mlflow.pytorch.log_model(
            pytorch_model=model,
            artifact_path="model",
            signature=signature,
            registered_model_name=MODEL_NAME
        )
        
        # Get the newly registered model version
        client = MlflowClient()
        latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
        latest_version = max([int(v.version) for v in latest_versions])
        
        # Set alias to "latest"
        client.set_registered_model_alias(MODEL_NAME, "latest", latest_version)
        
        print(f"✓ Model logged successfully!")
        print(f"  Model URI: {model_info.model_uri}")
        print(f"  Version: {latest_version}")
        print(f"  Alias: 'latest'")
        print(f"  Run ID: {run.info.run_id}")
        print(f"\n🎯 Model is now available for inference using alias: models:/{MODEL_NAME}@latest")

In [0]:
import mlflow
import mlflow.pytorch
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from mlflow.tracking import MlflowClient
from datetime import datetime
from pyspark.sql import functions as F

# ---- Model Registry Configuration ----
MODEL_NAME = "acubed.ffr.ffr_difficulty_model"  # Use acubed catalog

# ---- Load Model from MLflow using alias ----
print("Loading model from MLflow registry...")
model_uri = f"models:/{MODEL_NAME}@latest"
loaded_model = mlflow.pytorch.load_model(model_uri)
loaded_model.eval()

# Get model version info
client = MlflowClient()
model_version_info = client.get_model_version_by_alias(MODEL_NAME, "latest")
model_version_num = int(model_version_info.version)

print(f"Model loaded from: {model_uri}")
print(f"Model version: {model_version_num}")
print(f"Model max_difficulty: {loaded_model.max_difficulty}")

# ---- Load Data from Table (ALL songs, including difficulty 0) ----
print("\nLoading data from acubed.ffr.gold__features...")
df = spark.table('acubed.ffr.gold__features').toPandas()
# Do NOT filter out difficulty 0 here - we want predictions for all songs

# Define ALL 16 feature columns (updated to match gold layer schema)
FEATURE_COLS = [
    # Density Features (3)
    'vertical_density', 'horizontal_density', 'position_in_song',
    # Directional Complexity Features (4)
    'orientation_changed', 'is_single_arrow', 'double_jump', 'triple_quad_jump',
    # Temporal Pattern Features (3)
    'note_spacing', 'is_burst_note', 'is_isolated_note',
    # Song-Level Features (4)
    'song_length_log_normalized', 'vertical_density_cv', 'horizontal_density_cv', 'unique_orientations',
    # Game Mechanics (2)
    'hold', 'mine'
]

num_songs_total = df['song_id'].nunique()
num_songs_diff_0 = df[df['difficulty'] == 0]['song_id'].nunique()
num_songs_diff_gt_0 = df[df['difficulty'] > 0]['song_id'].nunique()

print(f"Loaded {len(df)} notes from {num_songs_total} songs")
print(f"  Songs with difficulty = 0: {num_songs_diff_0}")
print(f"  Songs with difficulty > 0: {num_songs_diff_gt_0}")
print(f"Using {len(FEATURE_COLS)} features: {FEATURE_COLS}")

# ---- Load song names from bronze table ----
print("\nLoading song names from acubed.ffr.bronze__songlist...")
song_names_df = spark.table('acubed.ffr.bronze__songlist').select('id', 'name').toPandas()
song_names_df = song_names_df.rename(columns={'id': 'song_id', 'name': 'song_name'})

print(f"Loaded {len(song_names_df)} song names from bronze table")

# Get song info with actual difficulty from features table
song_info = df.groupby('song_id').agg({
    'difficulty': 'first'
}).reset_index()

# Merge with song names from bronze table
song_info = song_info.merge(song_names_df, on='song_id', how='left')

print(f"Matched {song_info['song_name'].notna().sum()} songs with names")

# Group by song_id
song_groups = df.groupby('song_id')

# ---- Dataset Class (same as training) ----
class RealDataSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, song_groups):
        self.data = []
        self.labels = []
        self.lengths = []
        self.song_ids = []
        
        for song_id, group in song_groups:
            group_sorted = group.sort_values('note_id')
            seq = group_sorted[feature_cols].values.astype(np.float32)
            difficulty = group_sorted['difficulty'].iloc[0]
            
            self.data.append(seq)
            self.labels.append(difficulty)
            self.lengths.append(len(seq))
            self.song_ids.append(song_id)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return (torch.from_numpy(self.data[idx]), 
                torch.tensor(self.labels[idx], dtype=torch.float32), 
                self.lengths[idx])

def collate_fn(batch):
    seqs, ys, lengths = zip(*batch)
    max_len = max(lengths)
    padded_seqs = []
    mask = []
    for seq, l in zip(seqs, lengths):
        pad = torch.zeros(max_len - l, seq.shape[1])
        padded_seqs.append(torch.cat([seq, pad], dim=0))
        mask.append(torch.cat([torch.ones(l), torch.zeros(max_len - l)]))
    padded_seqs = torch.stack(padded_seqs)
    mask = torch.stack(mask)
    ys = torch.stack(ys)
    return padded_seqs, ys, mask

# ---- Create Dataset and DataLoader ----
print("\nPreparing dataset...")
dataset = RealDataSequenceDataset(df, FEATURE_COLS, song_groups)
full_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ---- Generate Predictions ----
print("\nGenerating predictions for all songs...")
all_preds = []
all_trues = []
all_song_ids = []

with torch.no_grad():
    batch_idx = 0
    for batch_seqs, batch_ys, batch_mask in full_loader:
        preds = loaded_model(batch_seqs, batch_mask)
        all_preds.extend(preds.cpu().numpy())
        all_trues.extend(batch_ys.cpu().numpy())
        
        start_idx = batch_idx * 32
        end_idx = min(start_idx + 32, len(dataset))
        all_song_ids.extend([dataset.song_ids[i] for i in range(start_idx, end_idx)])
        batch_idx += 1

# ---- Create Results DataFrame ----
results_df = pd.DataFrame({
    'song_id': all_song_ids,
    'actual_difficulty': all_trues,
    'predicted_difficulty': all_preds
})

# Merge with song names from bronze table
results_df = results_df.merge(song_info[['song_id', 'song_name']], on='song_id', how='left')

# Add model version and timestamp
results_df['model_version_num'] = model_version_num
results_df['computed_ts'] = datetime.now()

# Reorder columns
results_df = results_df[['song_id', 'song_name', 'actual_difficulty', 'predicted_difficulty', 
                         'model_version_num', 'computed_ts']]

print(f"\n{'='*60}")
print(f"PREDICTIONS GENERATED USING MODEL v{model_version_num}")
print(f"{'='*60}")
print(f"Total songs: {len(results_df)}")
print(f"  Songs with difficulty = 0: {len(results_df[results_df['actual_difficulty'] == 0])}")
print(f"  Songs with difficulty > 0: {len(results_df[results_df['actual_difficulty'] > 0])}")
print(f"Songs with names: {results_df['song_name'].notna().sum()}")

# Calculate metrics ONLY for songs with difficulty > 0
results_gt_0 = results_df[results_df['actual_difficulty'] > 0].copy()
error = results_gt_0['predicted_difficulty'] - results_gt_0['actual_difficulty']
abs_error = np.abs(error)

print(f"\nError Metrics (difficulty > 0 only, n={len(results_gt_0)}):")
print(f"  MAE:  {abs_error.mean():.2f} difficulty units")
print(f"  RMSE: {np.sqrt((error ** 2).mean()):.2f} difficulty units")
print(f"  Mean Error: {error.mean():.2f}")

# Display sample with both difficulty 0 and > 0
print("\nSample predictions (difficulty = 0):")
display(results_df[results_df['actual_difficulty'] == 0].head(10))

print("\nSample predictions (difficulty > 0):")
display(results_df[results_df['actual_difficulty'] > 0].head(10))

# Check specific songs 281 and 1405
print("\n" + "="*60)
print("SPECIFIC SONGS: 281 and 1405")
print("="*60)
specific_songs = results_df[results_df['song_id'].isin([281, 1405])]
if len(specific_songs) > 0:
    display(specific_songs)
else:
    print("⚠️ Songs 281 and/or 1405 not found in results")

# ---- Write to Delta Table (APPEND mode to track history) ----
print(f"\nAppending predictions to acubed.ffr.gold__predictions...")

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(results_df)

# Write to Delta table (append mode to keep version history)
spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("acubed.ffr.gold__predictions")

print(f"✓ Successfully appended {len(results_df)} predictions to acubed.ffr.gold__predictions")
print(f"  Model version: {model_version_num}")
print(f"  Timestamp: {results_df['computed_ts'].iloc[0]}")
print(f"  Songs with difficulty = 0: {len(results_df[results_df['actual_difficulty'] == 0])}")
print(f"  Songs with difficulty > 0: {len(results_df[results_df['actual_difficulty'] > 0])}")
print(f"\n💡 To view predictions over time, query by model_version_num and computed_ts")

In [0]:
%sql
WITH filtered_preds AS (
  SELECT *
  FROM acubed.ffr.gold__predictions
  WHERE NOT (
    model_version_num = 2 AND computed_ts = TIMESTAMP('2026-05-18T15:25:41.167+00:00')
  )
    AND actual_difficulty > 0
),
latest_per_version AS (
  SELECT
    song_id,
    model_version_num,
    actual_difficulty,
    predicted_difficulty,
    computed_ts,
    ROW_NUMBER() OVER (PARTITION BY song_id, model_version_num ORDER BY computed_ts DESC) AS rn
  FROM filtered_preds
),
pivoted AS (
  SELECT
    song_id,
    MAX(CASE WHEN model_version_num = 1 THEN predicted_difficulty END) AS pred_v1,
    MAX(CASE WHEN model_version_num = 2 THEN predicted_difficulty END) AS pred_v2,
    MAX(CASE WHEN model_version_num = 1 THEN actual_difficulty END) AS actual
  FROM latest_per_version
  WHERE rn = 1
  GROUP BY song_id
)
SELECT
  song_id,
  pred_v1,
  pred_v2,
  actual,
  ROUND(ABS(pred_v1 - actual), 3) AS v1_abs_error,
  ROUND(ABS(pred_v2 - actual), 3) AS v2_abs_error,
  ROUND((ABS(pred_v2 - actual) - ABS(pred_v1 - actual)), 3) AS error_change,
  CASE
    WHEN ABS(pred_v2 - actual) < ABS(pred_v1 - actual) THEN 'Improved'
    WHEN ABS(pred_v2 - actual) > ABS(pred_v1 - actual) THEN 'Worsened'
    ELSE 'Same'
  END AS convergence_direction
FROM pivoted
ORDER BY error_change ASC;

In [0]:
%sql
  SELECT *
  FROM acubed.ffr.gold__predictions order by song_id asc, model_version_num desc